In [0]:
%pip install statsmodels

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import confusion_matrix, classification_report

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.tree import (DecisionTreeClassifier as DTC,
                          DecisionTreeRegressor as DTR,
                          plot_tree,
                          export_text)
from sklearn.metrics import (accuracy_score,
                             log_loss)
from sklearn.ensemble import \
     (RandomForestRegressor as RF,
      GradientBoostingRegressor as GBR)


In [0]:
df = spark.table("eda.default.eda_silver_windowed").toPandas()
df.head()
# Replace 'malignant' with 1 and 'benign' with 0
df = df.query("condition != 'black'")
# Multiple categories become 1
df['condition'] = df['condition'].replace({
    'neutral': 0,
    'positive': 1,
    'negative': 1
})
print(df['condition'].value_counts(dropna=False))

In [0]:
#Do not do log, do z score for all variables
from sklearn.preprocessing import StandardScaler

# 1. Get a list of all columns EXCEPT 'condition'
cols_to_scale = df.select_dtypes(include=['number']).columns.difference(['condition', 'source_rows'])

# 2. Initialize and apply the scaler only to those columns
scaler = StandardScaler()
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

# Now df has standardized features, but 'condition' remains 0/1 (or its original state)

In [0]:
# 1. Define which columns are features (X)
# This excludes 'condition' so the model can't "cheat"
cols_to_scale = df.columns.difference(['condition', 'source_rows'])
X = df[cols_to_scale]

# 2. Define your target (y)
# Fixed the typo from 'condtion' to 'condition'
y = df['condition']

In [0]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import confusion_matrix, classification_report
import timeit
start_time = timeit.default_timer()

condition_col = 'condition'  # rename the variable to avoid collision
colnames = df.drop(columns=[condition_col, 'source_rows']).columns.tolist()

formula = f'{condition_col} ~ {" + ".join(colnames)}'
model = smf.glm(formula=formula, data=df, family=sm.families.Binomial())

result = model.fit()
print(result.summary())

# Print computational time and accuracy
elapsed = timeit.default_timer() - start_time
print("--- Run time is %s seconds ---" % elapsed)

In [0]:
%pip install lightgbm

import lightgbm as lgb
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import timeit

# Define start time to calculate computation time
start_time = timeit.default_timer()

y_train = df['condition']
Y = df['condition']

# Select only numeric columns from X (exclude 'index' if present)
X_numeric = X.select_dtypes(include=['number'])

# Initialize the LightGBM model
model = lgb.LGBMClassifier(learning_rate=0.09, max_depth=-5, random_state=42)

# Train the model
model.fit(X_numeric.values, y_train.values, eval_set=[(X_numeric.values, Y.values)], eval_metric='logloss')

# Predict on the test set
y_pred = model.predict(X_numeric.values)

# Calculate accuracy
accuracy = accuracy_score(Y, y_pred)


# Print computational time and accuracy
elapsed = timeit.default_timer() - start_time
print("--- Run time is %s seconds ---" % elapsed)
print(f"Accuracy on the test set: {accuracy:.2f}")

print(classification_report(Y, y_pred, digits=3))


In [0]:
from sklearn.model_selection import GroupKFold
import numpy as np
import lightgbm as lgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import timeit

# Define start time to calculate computation time
start_time = timeit.default_timer()

# Extract patient IDs from the index column
# Pattern: 'SUD_003_01_EDA_10s' -> patient ID is '01' (third component)
patient_ids = X['index'].str.split('_').str[2]

print(f"Total samples: {len(X)}")
print(f"Unique patients: {patient_ids.nunique()}")
print(f"Samples per patient (mean): {len(X) / patient_ids.nunique():.1f}")
print(f"\nUnique patient IDs: {sorted(patient_ids.unique().tolist())}")
print()

# Prepare features (exclude 'index' column), labels, and groups
X_features = X.select_dtypes(include=['number'])  # Only numeric columns
y_labels = Y
groups = patient_ids

# Initialize GroupKFold with 5 splits
gkf = GroupKFold(n_splits=5)

fold_accuracies = []
all_y_true = []
all_y_pred = []

print("Training with Group K-Fold Cross-Validation:")
print("=" * 60)

# Loop through the group-aware folds
for fold_num, (train_idx, val_idx) in enumerate(gkf.split(X_features, y_labels, groups=groups), 1):
    X_train_fold, X_val_fold = X_features.iloc[train_idx], X_features.iloc[val_idx]
    y_train_fold, y_val_fold = y_labels.iloc[train_idx], y_labels.iloc[val_idx]
    
    # Get patient IDs in this fold for verification
    train_patients = groups.iloc[train_idx].nunique()
    val_patients = groups.iloc[val_idx].nunique()
    
    # Initialize and train the LightGBM model
    model_fold = lgb.LGBMClassifier(learning_rate=0.09, max_depth=-5, random_state=42, verbose=-1)
    model_fold.fit(X_train_fold, y_train_fold, eval_set=[(X_val_fold, y_val_fold)], eval_metric='logloss')
    
    # Predict on validation set
    y_pred_fold = model_fold.predict(X_val_fold)
    
    # Calculate accuracy
    accuracy_fold = accuracy_score(y_val_fold, y_pred_fold)
    fold_accuracies.append(accuracy_fold)
    
    # Store predictions for overall confusion matrix
    all_y_true.extend(y_val_fold.tolist())
    all_y_pred.extend(y_pred_fold.tolist())
    
    # Confusion matrix for this fold
    cm_fold = confusion_matrix(y_val_fold, y_pred_fold)
    
    print(f"Fold {fold_num}:")
    print(f"  Train: {len(train_idx)} samples ({train_patients} patients)")
    print(f"  Val:   {len(val_idx)} samples ({val_patients} patients)")
    print(f"  Accuracy: {accuracy_fold:.4f}")
    print(f"  Confusion Matrix:")
    print(f"    [[TN={cm_fold[0,0]}, FP={cm_fold[0,1]}],")
    print(f"     [FN={cm_fold[1,0]}, TP={cm_fold[1,1]}]]")
    print()

# Print overall results
print("=" * 60)
print("Cross-Validation Results:")
print(f"  Mean Accuracy: {np.mean(fold_accuracies):.4f} (+/- {np.std(fold_accuracies):.4f})")
print(f"  Min Accuracy:  {np.min(fold_accuracies):.4f}")
print(f"  Max Accuracy:  {np.max(fold_accuracies):.4f}")

# Overall confusion matrix across all folds
print("\nAggregate Confusion Matrix (all folds):")
cm_overall = confusion_matrix(all_y_true, all_y_pred)
print(cm_overall)
print(f"\n              Predicted")
print(f"              0 (Neutral)  1 (Stimulated)")
print(f"Actual 0      {cm_overall[0,0]:<12} {cm_overall[0,1]:<14}")
print(f"Actual 1      {cm_overall[1,0]:<12} {cm_overall[1,1]:<14}")
print(f"\nTrue Negatives (TN):  {cm_overall[0,0]}")
print(f"False Positives (FP): {cm_overall[0,1]}")
print(f"False Negatives (FN): {cm_overall[1,0]}")
print(f"True Positives (TP):  {cm_overall[1,1]}")

# Classification report
print("\nClassification Report (all folds):")
print(classification_report(all_y_true, all_y_pred, target_names=['Neutral (0)', 'Stimulated (1)'], digits=4))

elapsed_gkf = timeit.default_timer() - start_time
print(f"\nTotal run time: {elapsed_gkf:.2f} seconds")

In [0]:
%pip install optuna

import optuna
from sklearn.model_selection import GroupKFold
import numpy as np
import lightgbm as lgb
from sklearn.metrics import accuracy_score
import timeit

# Define the objective function for Optuna
def objective(trial):
    # Suggest hyperparameters
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42,
        'verbose': -1
    }
    
    # Extract patient IDs
    patient_ids = X['index'].str.split('_').str[2]
    
    # Prepare features and labels
    X_features = X.select_dtypes(include=['number'])
    y_labels = Y
    groups = patient_ids
    
    # Initialize GroupKFold
    gkf = GroupKFold(n_splits=5)
    
    # Cross-validation scores
    fold_scores = []
    
    for train_idx, val_idx in gkf.split(X_features, y_labels, groups=groups):
        X_train_fold, X_val_fold = X_features.iloc[train_idx], X_features.iloc[val_idx]
        y_train_fold, y_val_fold = y_labels.iloc[train_idx], y_labels.iloc[val_idx]
        
        # Train model with suggested parameters
        model = lgb.LGBMClassifier(**params)
        model.fit(X_train_fold, y_train_fold, eval_set=[(X_val_fold, y_val_fold)], eval_metric='logloss')
        
        # Evaluate
        y_pred = model.predict(X_val_fold)
        score = accuracy_score(y_val_fold, y_pred)
        fold_scores.append(score)
    
    # Return mean accuracy across all folds
    return np.mean(fold_scores)

print("Starting Optuna Hyperparameter Optimization...")
print("=" * 60)
start_time = timeit.default_timer()

# Create study and optimize
study = optuna.create_study(direction='maximize', study_name='lgbm_optimization')
study.optimize(objective, n_trials=50, show_progress_bar=True)

elapsed_optuna = timeit.default_timer() - start_time

print("\n" + "=" * 60)
print("Optimization Complete!")
print(f"Total optimization time: {elapsed_optuna:.2f} seconds")
print(f"\nBest trial: {study.best_trial.number}")
print(f"Best accuracy: {study.best_value:.4f}")
print("\nBest hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

In [0]:
from sklearn.model_selection import GroupKFold
import numpy as np
import lightgbm as lgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import timeit

print("Training Final Model with Optimized Hyperparameters...")
print("=" * 60)
start_time = timeit.default_timer()

# Extract patient IDs
patient_ids = X['index'].str.split('_').str[2]

# Prepare features and labels
X_features = X.select_dtypes(include=['number'])
y_labels = Y
groups = patient_ids

# Initialize GroupKFold
gkf = GroupKFold(n_splits=5)

fold_accuracies = []
all_y_true = []
all_y_pred = []

# Use best parameters from Optuna
best_params = study.best_params.copy()
best_params['random_state'] = 42
best_params['verbose'] = -1

print(f"\nUsing optimized parameters:")
for key, value in best_params.items():
    print(f"  {key}: {value}")
print("\n" + "=" * 60)

# Loop through the group-aware folds with optimized parameters
for fold_num, (train_idx, val_idx) in enumerate(gkf.split(X_features, y_labels, groups=groups), 1):
    X_train_fold, X_val_fold = X_features.iloc[train_idx], X_features.iloc[val_idx]
    y_train_fold, y_val_fold = y_labels.iloc[train_idx], y_labels.iloc[val_idx]
    
    # Get patient IDs in this fold
    train_patients = groups.iloc[train_idx].nunique()
    val_patients = groups.iloc[val_idx].nunique()
    
    # Train with optimized parameters
    model_optimized = lgb.LGBMClassifier(**best_params)
    model_optimized.fit(X_train_fold, y_train_fold, eval_set=[(X_val_fold, y_val_fold)], eval_metric='logloss')
    
    # Predict
    y_pred_fold = model_optimized.predict(X_val_fold)
    
    # Calculate accuracy
    accuracy_fold = accuracy_score(y_val_fold, y_pred_fold)
    fold_accuracies.append(accuracy_fold)
    
    # Store predictions
    all_y_true.extend(y_val_fold.tolist())
    all_y_pred.extend(y_pred_fold.tolist())
    
    # Confusion matrix for this fold
    cm_fold = confusion_matrix(y_val_fold, y_pred_fold)
    
    print(f"Fold {fold_num}:")
    print(f"  Train: {len(train_idx)} samples ({train_patients} patients)")
    print(f"  Val:   {len(val_idx)} samples ({val_patients} patients)")
    print(f"  Accuracy: {accuracy_fold:.4f}")
    print(f"  Confusion Matrix: [[TN={cm_fold[0,0]}, FP={cm_fold[0,1]}], [FN={cm_fold[1,0]}, TP={cm_fold[1,1]}]]")
    print()

# Print overall results
print("=" * 60)
print("Optimized Model Results:")
print(f"  Mean Accuracy: {np.mean(fold_accuracies):.4f} (+/- {np.std(fold_accuracies):.4f})")
print(f"  Min Accuracy:  {np.min(fold_accuracies):.4f}")
print(f"  Max Accuracy:  {np.max(fold_accuracies):.4f}")

# Overall confusion matrix
print("\nAggregate Confusion Matrix (all folds):")
cm_overall = confusion_matrix(all_y_true, all_y_pred)
print(cm_overall)
print(f"\n              Predicted")
print(f"              0 (Neutral)  1 (Stimulated)")
print(f"Actual 0      {cm_overall[0,0]:<12} {cm_overall[0,1]:<14}")
print(f"Actual 1      {cm_overall[1,0]:<12} {cm_overall[1,1]:<14}")
print(f"\nTrue Negatives (TN):  {cm_overall[0,0]}")
print(f"False Positives (FP): {cm_overall[0,1]}")
print(f"False Negatives (FN): {cm_overall[1,0]}")
print(f"True Positives (TP):  {cm_overall[1,1]}")

# Classification report
print("\nClassification Report (all folds):")
print(classification_report(all_y_true, all_y_pred, target_names=['Neutral (0)', 'Stimulated (1)'], digits=4))

elapsed_final = timeit.default_timer() - start_time
print(f"\nFinal model training time: {elapsed_final:.2f} seconds")

# Compare with baseline
print("\n" + "=" * 60)
print("Comparison with Baseline:")
print(f"  Baseline accuracy (from Cell 8): 0.6154")
print(f"  Optimized accuracy: {np.mean(fold_accuracies):.4f}")
print(f"  Improvement: {(np.mean(fold_accuracies) - 0.6154):.4f} ({(np.mean(fold_accuracies) - 0.6154) / 0.6154 * 100:+.2f}%)")

In [0]:
import pandas as pd
import matplotlib.pyplot as plt

# Extract feature importances from the final optimized model
feat_imp = pd.DataFrame({
    'feature': X_features.columns,
    'importance': model_optimized.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 20 Most Important Features:")
print("=" * 60)
print(feat_imp.head(20).to_string(index=False))

# Visualize top 15 features
plt.figure(figsize=(10, 8))
plt.barh(feat_imp.head(15)['feature'], feat_imp.head(15)['importance'])
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Top 15 Most Important Features (Optimized Model)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\nTotal features: {len(feat_imp)}")
print(f"Features with importance > 0: {(feat_imp['importance'] > 0).sum()}")